# Training and validation

In [1]:
import torch
from Utils import AnaliseGraficaVal, train, eval, fix_random_seed, predict
# Importe a classe que criamos anteriormente
from Datasets import Eucalyptus_closedset_loader
from Modelos.ResNet18_32x32_backbone import ResNet18_32x32
from Modelos.AlexNet_backbone import Alexnet
from torchvision import transforms
from torchvision.models import AlexNet_Weights
import torch.nn as nn
import torch.optim as optim
from Utils.Nomes import NOMES
import gc
import os

# Configurações iniciais
fix_random_seed(42)
device = "cuda:0" if torch.cuda.is_available() else "cpu"

lr = 0.0001
epochs = 25
bs = 128
num_classes = 3
n_folds = 5
dataset= "dataset-1"

weights = AlexNet_Weights.IMAGENET1K_V1


# 2. Inicialização do Loader customizado
data_manager = Eucalyptus_closedset_loader(bs=bs,dataset=dataset)
model_name = "AlexNet"
model_dir = f"/home/alexandreselani/Desktop/Eucalyptus/ClosedSet/Models/{dataset}/"


In [2]:

for fold in range(n_folds):
    torch.cuda.empty_cache()
    
    gc.collect()

    grafico = AnaliseGraficaVal(f"{model_name}_fold_{fold}", "Eucalyptus closed set", dir=f"/home/alexandreselani/Desktop/Eucalyptus/ClosedSet/Graficos/{dataset}/")

    # 4. Modelo, Critério e Otimizador
    model = Alexnet(num_classes,weights=weights)
    model = model.to(device)

    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    criterion = torch.nn.CrossEntropyLoss()

    train_dataloader = data_manager.load_train(transform=weights.transforms(),fold=fold)
    
    val_dataloader = data_manager.load_val(transform=weights.transforms(),fold=fold)

    for epoch in range(epochs):
        model.train()
        train_loss, train_acc = train(train_dataloader, model, criterion, optimizer)
        
        model.eval()
        val_loss, val_acc = eval(val_dataloader, model, criterion)
        
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | lr = {optimizer.param_groups[0]['lr']:.6f}")
        print(f"VALIDATION    | Loss: {val_loss:.4f} | Acc: {val_acc:.4f}")

        grafico.addEpochVal(epoch, train_loss, train_acc, val_loss, val_acc)

    grafico.mostraGraficoVal()

    # 7. Salvamento
   
    
    os.makedirs(model_dir,exist_ok=True)
    torch.save(model.state_dict(), f"{model_dir}AlexNet_fold_{fold}.pt")
    print(f"Modelo salvo em {model_dir}")

    del model,optimizer,criterion,grafico,train_dataloader,val_dataloader

Epoch 1/25 | Train Loss: 0.3871 | Acc: 0.8217 | lr = 0.000100
VALIDATION    | Loss: 0.1701 | Acc: 0.9695
Epoch 2/25 | Train Loss: 0.0976 | Acc: 0.9694 | lr = 0.000100
VALIDATION    | Loss: 0.1122 | Acc: 0.9695
Epoch 3/25 | Train Loss: 0.0639 | Acc: 0.9754 | lr = 0.000100
VALIDATION    | Loss: 0.0809 | Acc: 0.9790
Epoch 4/25 | Train Loss: 0.0455 | Acc: 0.9885 | lr = 0.000100
VALIDATION    | Loss: 0.0621 | Acc: 0.9866
Epoch 5/25 | Train Loss: 0.0331 | Acc: 0.9930 | lr = 0.000100
VALIDATION    | Loss: 0.0480 | Acc: 0.9924
Epoch 6/25 | Train Loss: 0.0264 | Acc: 0.9949 | lr = 0.000100
VALIDATION    | Loss: 0.0369 | Acc: 0.9924
Epoch 7/25 | Train Loss: 0.0215 | Acc: 0.9947 | lr = 0.000100
VALIDATION    | Loss: 0.0279 | Acc: 0.9924
Epoch 8/25 | Train Loss: 0.0173 | Acc: 0.9968 | lr = 0.000100
VALIDATION    | Loss: 0.0202 | Acc: 0.9981
Epoch 9/25 | Train Loss: 0.0143 | Acc: 0.9968 | lr = 0.000100
VALIDATION    | Loss: 0.0162 | Acc: 0.9981
Epoch 10/25 | Train Loss: 0.0127 | Acc: 0.9968 | lr = 0

KeyboardInterrupt: 

# Testing

In [2]:
import numpy as np
from sklearn.metrics import accuracy_score,confusion_matrix,ConfusionMatrixDisplay,roc_auc_score
from Utils import AnaliseGraficaVal, train, eval, fix_random_seed, predict,metricasImplementadas
import pandas as pd
import torch.nn.functional as F

cm = np.zeros((num_classes,num_classes),dtype=int)

results = {'f1': [], 'acc': [], 'uuc_acc': [], 'inner': [], 'outer': [], 'half': [], 'auroc': []}
for fold in range(n_folds):
    model = Alexnet(num_classes=num_classes)
    model.load_state_dict(torch.load(f"{model_dir}AlexNet_fold_{fold}.pt"))
    model.to(device)
    test_dataloader = data_manager.load_test(fold,transform=weights.transforms())

    y_true = []
    for _,y in test_dataloader:
        y_true.append(y)

    #print(y_true)
    y_true = torch.concat(y_true)
    y_pred,scores,outputs = predict(test_dataloader,model)

    
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()
    scores = scores.cpu()
    outputs = outputs.cpu()
    print(y_true)
    
    metrics = metricasImplementadas(label=y_true,predict=y_pred,metodo="closedset")._metricas()

    results['f1'].append(metrics["F1 macro"])
    results['acc'].append(metrics["accuracy"][0])
    results['uuc_acc'].append(metrics["UUC Accuracy"][0])
    results['inner'].append(metrics["inner metric"][0])
    results['outer'].append(metrics["outer metric"][0])
    results['half'].append(metrics["halfpoint"][0])
    results['auroc'].append(roc_auc_score(y_true=y_true,y_score=F.softmax(outputs),multi_class="ovo"))
    
    cm = cm + confusion_matrix(y_true,y_pred)

final_data = []


final_data.append({
    "f1_macro_mean": np.mean(results['f1']),
    "f1_macro_std": np.std(results['f1']),
    "acc_mean": np.mean(results['acc']),
    "acc_std": np.std(results['acc']),
    "uuc_acc_mean": np.mean(results['uuc_acc']),
    "uuc_acc_std": np.std(results['uuc_acc']),
    "inner_mean": np.mean(results['inner']),
    "inner std": np.std(results["inner"]),
    "outer_mean": np.mean(results['outer']),
    "outer_std": np.std(results["outer"]),
    "halfpoint_mean": np.mean(results['half']),
    "halfpoint_std": np.std(results['half']),
    "auroc_mean": np.mean(results['auroc']),
    "auroc_std": np.std(results['auroc'])
})

df_results = pd.DataFrame(final_data)
df_results.to_csv(f"test_closedset_eucalyptus_{dataset}.csv", index=False, float_format="%.3f") 


tensor([[ 3.9988, -3.4404,  0.0744],
        [ 3.9802, -3.2909, -0.2388],
        [ 3.9098, -2.7695, -0.6962],
        [ 3.7802, -3.5731,  0.5105],
        [ 4.4856, -3.0907, -1.0095],
        [ 3.8099, -2.7995, -0.6106],
        [ 4.4870, -3.4698, -0.5930],
        [ 3.5257, -2.3153, -0.7049],
        [ 4.0196, -3.1375, -0.3919],
        [ 4.4295, -3.2170, -0.8259],
        [ 4.1558, -2.7138, -1.2503],
        [ 3.9652, -3.5411,  0.0700],
        [ 4.2497, -2.9304, -0.9732],
        [ 4.3554, -2.7696, -0.9868],
        [ 4.4277, -3.0725, -0.8917],
        [ 4.3285, -2.6844, -1.3215],
        [ 4.5533, -3.0944, -0.9658],
        [ 4.5160, -3.0464, -1.0422],
        [ 3.6157, -2.3611, -1.0208],
        [ 4.2097, -2.9356, -0.7868],
        [ 4.3835, -2.8127, -1.2779],
        [ 4.4501, -2.8607, -1.2672],
        [ 4.2239, -2.9283, -1.0674],
        [ 4.3067, -2.8843, -1.2128],
        [ 3.7721, -2.5311, -0.8145],
        [ 4.1475, -3.3911, -0.2289],
        [ 4.2957, -2.7558, -1.2966],
 

/tmp/ipykernel_490600/2258688487.py:39: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  results['auroc'].append(roc_auc_score(y_true=y_true,y_score=F.softmax(outputs),multi_class="ovo"))


tensor([[ 3.8351, -2.6634, -2.5035],
        [ 3.5704, -2.7480, -2.1888],
        [ 3.0328, -2.9378, -1.3979],
        [ 3.4198, -2.8012, -1.9032],
        [ 3.8151, -3.0101, -2.3958],
        [ 3.8911, -2.7305, -2.5396],
        [ 3.8243, -2.7272, -2.4533],
        [ 2.7877, -2.8053, -1.2802],
        [ 3.5302, -2.8449, -2.0935],
        [ 3.9870, -2.9829, -2.3528],
        [ 3.8389, -3.0427, -2.2539],
        [ 3.8175, -2.8976, -2.3672],
        [ 2.9164, -3.3112, -1.2461],
        [ 3.8310, -2.9398, -2.3018],
        [ 3.8291, -2.5893, -2.6484],
        [ 3.6687, -2.9806, -2.1309],
        [ 2.3624, -3.6057, -0.4032],
        [ 3.6519, -3.0672, -2.0704],
        [ 1.7074, -2.3996, -0.6257],
        [ 3.9109, -2.7083, -2.5146],
        [ 3.6458, -2.9888, -1.9964],
        [ 3.5860, -3.1126, -2.0268],
        [ 3.2024, -2.7419, -1.7354],
        [ 3.5208, -2.8428, -2.1931],
        [ 3.8199, -2.8288, -2.3562],
        [ 2.9959, -2.4911, -1.7177],
        [ 0.8586, -1.6305, -0.3482],
 

/tmp/ipykernel_490600/2258688487.py:39: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  results['auroc'].append(roc_auc_score(y_true=y_true,y_score=F.softmax(outputs),multi_class="ovo"))


tensor([[ 4.1575, -3.0128, -0.7117],
        [ 3.4293, -1.9062, -0.7021],
        [ 4.4162, -2.7778, -1.1830],
        [ 4.3534, -2.8284, -1.1339],
        [ 3.9620, -3.0498, -0.5639],
        [ 4.3888, -2.8496, -1.1177],
        [ 4.2818, -2.6385, -1.1540],
        [ 4.2701, -2.4197, -1.3652],
        [ 3.4324, -2.3556, -0.3579],
        [ 3.2204, -2.5074, -0.1452],
        [ 3.9962, -2.2667, -1.1096],
        [ 4.4155, -2.8747, -1.0713],
        [ 4.3226, -2.6878, -1.2004],
        [ 3.3796, -2.6323, -0.3016],
        [ 1.2549, -3.7664,  2.3764],
        [ 3.9771, -3.1309, -0.3641],
        [ 4.1605, -3.1714, -0.4346],
        [ 3.1463, -2.0526, -0.3152],
        [ 4.1556, -3.1429, -0.5913],
        [ 4.2923, -3.0281, -0.8549],
        [ 4.1808, -2.5103, -1.2128],
        [ 4.4026, -2.7392, -1.1784],
        [ 3.6011, -2.4693, -0.5274],
        [ 4.3876, -2.8018, -1.1418],
        [ 4.4138, -2.8643, -1.1250],
        [ 4.1829, -3.0415, -0.7116],
        [ 4.2945, -3.0590, -0.8183],
 

/tmp/ipykernel_490600/2258688487.py:39: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  results['auroc'].append(roc_auc_score(y_true=y_true,y_score=F.softmax(outputs),multi_class="ovo"))


tensor([[ 4.0453, -2.8673, -1.1260],
        [ 3.7983, -2.4846, -1.2156],
        [ 2.6516, -2.4002, -0.0419],
        [ 4.2113, -3.3530, -0.8007],
        [ 4.1335, -3.1365, -0.9141],
        [ 4.1620, -2.8659, -1.1729],
        [ 3.4954, -2.7605, -0.5910],
        [ 4.2228, -3.3164, -0.7458],
        [ 4.2562, -2.9994, -1.1407],
        [ 3.1776, -2.2091, -1.0064],
        [ 4.0361, -2.6265, -1.2612],
        [ 3.8456, -2.3468, -1.1779],
        [ 4.0713, -2.9274, -1.0443],
        [ 4.1333, -3.3201, -0.6885],
        [ 3.7691, -2.7609, -0.8151],
        [ 4.0970, -2.8045, -1.2321],
        [ 4.1237, -3.1854, -0.9318],
        [ 4.1322, -3.1206, -0.9455],
        [ 3.9017, -2.5896, -0.9717],
        [ 3.9324, -3.2217, -0.2655],
        [ 4.1766, -2.8015, -1.1907],
        [ 4.2635, -3.0980, -1.0634],
        [ 3.5004, -3.3310,  0.2424],
        [ 3.9605, -3.0466, -0.5882],
        [ 4.1379, -3.1232, -0.9746],
        [ 4.2184, -2.9832, -1.1003],
        [ 3.6005, -3.4500, -0.0362],
 

/tmp/ipykernel_490600/2258688487.py:39: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  results['auroc'].append(roc_auc_score(y_true=y_true,y_score=F.softmax(outputs),multi_class="ovo"))


tensor([[ 3.0359, -2.3898, -1.4184],
        [ 3.5729, -2.2042, -2.0151],
        [ 2.3377, -1.9480, -0.9867],
        [ 3.4284, -3.0771, -1.4286],
        [ 3.8074, -2.1533, -2.5383],
        [ 3.1588, -2.4930, -1.5141],
        [ 3.1036, -2.7534, -1.4717],
        [ 2.5849, -2.5551, -0.9811],
        [ 2.8403, -2.3172, -1.4184],
        [ 3.2822, -2.9335, -1.3497],
        [ 3.8153, -2.0816, -2.5562],
        [ 3.6183, -2.2035, -2.4348],
        [ 3.7108, -2.4734, -2.1303],
        [ 3.8806, -2.4066, -2.3955],
        [ 3.1743, -2.7756, -1.5555],
        [ 3.7753, -2.5265, -2.3094],
        [ 3.7815, -2.0722, -2.5716],
        [ 3.7185, -2.6071, -2.0592],
        [ 2.7846, -2.5235, -1.1501],
        [ 3.8260, -2.1298, -2.5906],
        [ 3.1209, -3.0872, -1.2627],
        [ 3.2337, -2.7811, -1.4951],
        [ 3.7803, -2.5910, -2.0751],
        [ 3.4528, -2.4161, -2.1146],
        [ 3.8286, -2.5979, -2.0870],
        [ 3.7973, -2.4430, -2.2503],
        [ 3.6000, -2.9895, -1.6154],
 

/tmp/ipykernel_490600/2258688487.py:39: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  results['auroc'].append(roc_auc_score(y_true=y_true,y_score=F.softmax(outputs),multi_class="ovo"))


In [3]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=[c for c in range(num_classes)])
disp.plot(ax=ax)
plt.title(f"Confusion matrix - {model_name}: {dataset}")
image_name = f"Confusion matrix - {model_name}: {dataset}.png"
plt.savefig(image_name, dpi=300, bbox_inches='tight')
plt.close(fig)